In [ ]:
import json
import gzip
import os

from langchain import hub
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.schema import Document
from tqdm import tqdm


In [ ]:
main_docs_only = True  # Set to True if you want to filter for main documents only
data_folder = "data"  # Specify the folder where your data is stored


train_file = f"train_no_ne.json.gz"
with gzip.open(os.path.join(data_folder, train_file), "r") as f:
    docs = json.load(f)

with open(os.path.join(data_folder, "docs_per_entity.json"), "r") as f:
    docs_per_entity = json.load(f)

main_docs = None
if main_docs_only:
    with open(os.path.join(data_folder, "main_doc_ids.json"), "r") as f:
        main_docs = json.load(f)

In [ ]:
UNAVAILABLE_DOCS = [
    "en_simple_wiki_v0-0001.json.gz-56175878",
    "en_simple_wiki_v0-0000.json.gz-32808961",
    "en_simple_wiki_v0-0001.json.gz-72498864",
    "en_simple_wiki_v0-0001.json.gz-73213510",
    "en_simple_wiki_v0-0001.json.gz-45577729",
    "en_simple_wiki_v0-0001.json.gz-11277190",
    "en_simple_wiki_v0-0001.json.gz-930826",
    "en_simple_wiki_v0-0000.json.gz-20006714",
    "en_simple_wiki_v0-0000.json.gz-16401475",
    "en_simple_wiki_v0-0000.json.gz-9934421",
    "en_simple_wiki_v0-0000.json.gz-30575720",
    "en_simple_wiki_v0-0000.json.gz-7728160",
    "en_simple_wiki_v0-0001.json.gz-62034570",
    "en_simple_wiki_v0-0001.json.gz-37567473",
    "en_simple_wiki_v0-0000.json.gz-25416724",
]


def load_relevant_docs(docs, docs_per_entity, main_docs=None):
    relevant_docs = {}
    for entity, doc_ids in tqdm(docs_per_entity.items()):
        relevant_docs[entity] = []
        for doc_id in doc_ids:
            if main_docs is not None and doc_id != main_docs[entity]:
                continue
            if doc_id not in UNAVAILABLE_DOCS:
                doc = docs.get(doc_id)
                relevant_docs[entity] += [
                    Document(
                        page_content=sent, metadata={"doc_id": doc_id, "entity": entity}
                    )
                    for sent_id, sent in enumerate(doc)
                ]

    return relevant_docs

In [ ]:
relevant_docs = load_relevant_docs(docs, docs_per_entity, main_docs)

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
with open("data/validation_probes.json", "r") as f:
    validation_probes = json.load(f)

## Gold Document

In [ ]:
validation_probes_w_rag = {}
k = 1  # Number of contexts to retrieve for each question

for entity, question_types in tqdm(
    validation_probes.items(), desc="Extracting Passage with highest similarity"
):
    vector_store = Chroma.from_documents(
        documents=relevant_docs[entity],
        embedding=embeddings,
        # collection_name="example_collection",
        persist_directory=None,  # Set to None if you don't want to persist
        # persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
    )
    for question_type, questions in question_types.items():
        if entity not in validation_probes_w_rag:
            validation_probes_w_rag[entity] = {}
        if question_type in [
            "Factual Question",
            "Time-Sensitive Question",
            "Temporal Understanding Question",
            "Entity Linking Question",
            "Rephrased Questions",
        ]:
            for question in questions:
                text = question["question"]
                context = vector_store.similarity_search(text, k=k)

                rag_question = question.copy()

                context = "\n".join([doc.page_content for doc in context])
                rag_question["question"] = f"Context: {context}\n{text}"

                if question_type not in validation_probes_w_rag[entity]:
                    validation_probes_w_rag[entity][question_type] = []

                validation_probes_w_rag[entity][question_type].append(rag_question)
        else:
            validation_probes_w_rag[entity][question_type] = questions
    vector_store.delete_collection()

In [ ]:
with open("data/validation_probes_w_rag_gold.json", "w") as f:
    json.dump(validation_probes_w_rag, f, indent=4)

## Main Documents

In [ ]:
validation_probes_w_rag = {}
k = 1  # Number of contexts to retrieve for each question

vector_store = Chroma.from_documents(
    documents=[doc for entity_docs in relevant_docs.values() for doc in entity_docs],
    embedding=embeddings,
    # collection_name="example_collection",
    persist_directory=None,  # Set to None if you don't want to persist
    # persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

print("# passages in vector store:", vector_store._collection.count())

questions_with_correct_doc = 0
total_questions = 0

for entity, question_types in tqdm(
    validation_probes.items(), desc="Extracting Passage with highest similarity"
):
    for question_type, questions in question_types.items():
        if entity not in validation_probes_w_rag:
            validation_probes_w_rag[entity] = {}
        if question_type in [
            "Factual Question",
            "Time-Sensitive Question",
            "Temporal Understanding Question",
            "Entity Linking Question",
            "Rephrased Questions",
        ]:
            for question in questions:
                text = question["question"]
                context = vector_store.similarity_search(text, k=k)

                rag_question = question.copy()

                metadata = [doc.metadata["entity"] for doc in context]
                context = "\n".join([doc.page_content for doc in context])

                rag_question["question"] = f"Context: {context}\n{text}"
                rag_question["metadata"] = metadata

                if entity in metadata:
                    questions_with_correct_doc += 1

                total_questions += 1


                if question_type not in validation_probes_w_rag[entity]:
                    validation_probes_w_rag[entity][question_type] = []

                validation_probes_w_rag[entity][question_type].append(rag_question)
        else:
            validation_probes_w_rag[entity][question_type] = questions

In [ ]:
with open("data/validation_probes_w_rag_main.json", "w") as f:
    json.dump(validation_probes_w_rag, f, indent=4)

In [ ]:
print(f"Questions with correct doc: {questions_with_correct_doc}/{total_questions} ({questions_with_correct_doc/total_questions:.2%})")

## All Documents

In [ ]:
relevant_docs = load_relevant_docs(docs, docs_per_entity)

In [ ]:
unique_docs = set()
unique_doc_list = []
for entity_docs in relevant_docs.values():
    doc_id = entity_docs[0].metadata["doc_id"]
    if doc_id not in unique_docs:
        unique_docs.add(doc_id)
        unique_doc_list.extend(entity_docs)

print(f"Number of unique documents: {len(unique_doc_list)}")
print(f"Number of total documents: {sum(len(docs) for docs in relevant_docs.values())}")

In [ ]:
vector_store = Chroma.from_documents(
    documents=unique_doc_list,
    embedding=embeddings,
    # collection_name="example_collection",
    persist_directory=None,  # Set to None if you don't want to persist
    # persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

print("# passages in vector store:", vector_store._collection.count())

In [ ]:
validation_probes_w_rag = {}
k = 1  # Number of contexts to retrieve for each question

questions_with_correct_doc = 0
total_questions = 0

for entity, question_types in tqdm(
    validation_probes.items(), desc="Extracting Passage with highest similarity"
):
    for question_type, questions in question_types.items():
        if entity not in validation_probes_w_rag:
            validation_probes_w_rag[entity] = {}
        if question_type in [
            "Factual Question",
            "Time-Sensitive Question",
            "Temporal Understanding Question",
            "Entity Linking Question",
            "Rephrased Questions",
        ]:
            for question in questions:
                main_doc_id = main_docs[entity] if main_docs is not None else None
                text = question["question"]
                context = vector_store.similarity_search(text, k=k)

                rag_question = question.copy()

                metadata = [doc.metadata["doc_id"] for doc in context]
                context = "\n".join([doc.page_content for doc in context])

                rag_question["question"] = f"Context: {context}\n{text}"
                rag_question["metadata"] = metadata

                if main_doc_id in metadata:
                    questions_with_correct_doc += 1

                total_questions += 1


                if question_type not in validation_probes_w_rag[entity]:
                    validation_probes_w_rag[entity][question_type] = []

                validation_probes_w_rag[entity][question_type].append(rag_question)
        else:
            validation_probes_w_rag[entity][question_type] = questions

In [ ]:
with open("data/validation_probes_w_rag_all.json", "w") as f:
    json.dump(validation_probes_w_rag, f, indent=4)

In [ ]:
print(f"Questions with correct doc: {questions_with_correct_doc}/{total_questions} ({questions_with_correct_doc/total_questions:.2%})")